<div style="font-size:30px;font-weight:700;color:#111827;padding-bottom:8px;margin:18px 0;">
AI Agent 개념 실습
</div>

로컬 GPU에 올린 **Qwen2.5-0.5B-Instruct(4bit)** 모델을 LangChain의 커스텀 ChatModel로 감싸고,
LLM이 스스로 **도구(Tool)를 골라 사용하는 Agent**의 핵심 개념을 단계별로 직접 구현해 봅니다.

| 단계 | 내용 |
|---|---|
| Tool | `@tool` 데코레이터로 도구 정의, Chroma 기반 문서 검색(RAG) 도구 |
| ReAct | Thought → Action → Observation 루프를 직접 구현 |
| Conversational | 대화 기록으로 맥락을 유지하는 에이전트 |
| Planning | 계획을 먼저 세우고 단계별로 실행하는 Plan-and-Execute |

# 기본환경 설정

In [ ]:
# %%capture
# %pip install -q -U unsloth langchain-core langchain_text_splitters langchain_huggingface langchain-chroma

# GPU와 실행 환경 확인

런타임 유형이 **GPU(T4)** 로 설정되어 있는지 확인합니다.

In [ ]:
import torch

print(f"PyTorch 버전: {torch.__version__}")
print(f"사용 GPU: {torch.cuda.get_device_name(0)}")

# 4비트 Qwen2.5 모델 로딩

미리 4bit로 양자화된 모델이라 Colab 무료 T4에서도 가볍게 동작합니다.
공개 모델이므로 Hugging Face 로그인은 필요하지 않습니다.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
)

model.eval()

# 커스텀 ChatModel (QwenChatModel)

In [ ]:
import torch
from threading import Thread
from typing import Any, Iterator, List, Optional
from pydantic import ConfigDict

In [ ]:
from langchain_core.callbacks import CallbackManagerForLLMRun
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, AIMessageChunk, BaseMessage, HumanMessage, SystemMessage
from langchain_core.outputs import ChatGeneration, ChatGenerationChunk, ChatResult

from transformers import TextIteratorStreamer

In [ ]:
class QwenChatModel(BaseChatModel):
    """Qwen2.5-Instruct를 LangChain ChatModel로 감싸는 클래스."""

    # BaseChatModel은 Pydantic 기반이므로 필드 선언이 필요합니다.
    model: Any
    tokenizer: Any

    max_tokens: int = 512
    do_sample: bool = True
    temperature: float = 0.7
    top_p: float = 0.9

    model_config = ConfigDict(arbitrary_types_allowed=True)

    @property
    def _llm_type(self) -> str:
        return "qwen2.5-custom-chatmodel"

    def _tokenize(self, messages: List[BaseMessage]):
        """LangChain 메시지를 Qwen 채팅 형식으로 변환하고 토큰화합니다."""
        chat = []

        for message in messages:
            if isinstance(message, SystemMessage):
                role = "system"
            elif isinstance(message, HumanMessage):
                role = "user"
            elif isinstance(message, AIMessage):
                role = "assistant"
            else:
                role = "user"

            chat.append({"role": role, "content": message.content})

        inputs = self.tokenizer.apply_chat_template(
            chat,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
        )
        return inputs.to(self.model.device)

    def _generation_options(self):
        """model.generate()에 공통으로 전달할 옵션입니다."""
        # Agent는 단계가 진행될수록 프롬프트가 길어지므로
        # "새로 생성할 토큰 수"를 제한하는 max_new_tokens를 사용합니다.
        options = {
            "max_length": self.max_tokens,
            "do_sample": self.do_sample,
            "pad_token_id": self.tokenizer.pad_token_id,
        }

        if self.do_sample:
            options["temperature"] = self.temperature
            options["top_p"] = self.top_p

        return options

    def _cut_at_stop(self, text: str, stop: Optional[List[str]]) -> str:
        """stop 문자열이 나오면 그 앞에서 텍스트를 잘라냅니다."""
        if stop:
            for s in stop:
                idx = text.find(s)
                if idx != -1:
                    text = text[:idx]
        return text.strip()

    def _generate(
        self,
        messages: List[BaseMessage],
        stop: Optional[List[str]] = None,
        run_manager: Optional[CallbackManagerForLLMRun] = None,
        **kwargs: Any,
    ) -> ChatResult:
        """invoke()와 batch()가 사용하는 메서드입니다."""
        inputs = self._tokenize(messages)
        input_length = inputs["input_ids"].shape[1]

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                **self._generation_options(),
            )

        new_tokens = outputs[0][input_length:]
        text = self.tokenizer.decode(
            new_tokens,
            skip_special_tokens=True,
        ).strip()

        text = self._cut_at_stop(text, stop)

        return ChatResult(
            generations=[ChatGeneration(message=AIMessage(content=text))]
        )

    def _stream(
        self,
        messages: List[BaseMessage],
        stop: Optional[List[str]] = None,
        run_manager: Optional[CallbackManagerForLLMRun] = None,
        **kwargs: Any,
    ) -> Iterator[ChatGenerationChunk]:
        """stream()이 사용하는 메서드입니다."""
        inputs = self._tokenize(messages)

        streamer = TextIteratorStreamer(
            self.tokenizer,
            skip_prompt=True,
            skip_special_tokens=True,
        )

        thread = Thread(
            target=self.model.generate,
            kwargs={
                **inputs,
                **self._generation_options(),
                "streamer": streamer,
            },
        )
        thread.start()

        for text in streamer:
            chunk = ChatGenerationChunk(
                message=AIMessageChunk(content=text)
            )

            if run_manager:
                run_manager.on_llm_new_token(text, chunk=chunk)

            yield chunk

        thread.join()

## Structured Output

Agent가 도구를 쓰거나 계획을 세우려면, LLM의 출력이 **정해진 형식(구조)** 을 따라야 합니다.
`PydanticOutputParser`는 Pydantic 모델의 스키마를 프롬프트에 넣어 주고,
모델이 출력한 JSON을 다시 Pydantic 객체로 변환해 줍니다.

> 형식이 중요한 작업에서는 `do_sample=False`(greedy) 모델을 쓰는 것이 훨씬 안정적입니다.
> 이 결정적(deterministic) 모델은 뒤의 Agent 실습에서도 계속 사용합니다.

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
# 형식이 중요한 작업(구조화 출력, Agent)용 결정적 모델
det_model = QwenChatModel(model=model, tokenizer=tokenizer, max_tokens=2048, do_sample=False)

In [ ]:
class MovieInfo(BaseModel):
    title: str = Field(..., description="영화 제목")
    year: int = Field(..., description="개봉 연도")
    genres: list[str] = Field(..., description="장르 목록")
    rating: float = Field(..., description="10점 만점 평점")

parser = PydanticOutputParser(pydantic_object=MovieInfo)

# 파서가 만들어 주는 형식 안내문을 확인해 봅니다.
print(parser.get_format_instructions())

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "요청받은 정보를 **JSON으로만 출력하세요**. 다른 설명은 쓰지 마세요.\n -출력형식\n{format_instructions}"),
    ("human", "{question}"),
]).partial(format_instructions=parser.get_format_instructions())

In [ ]:
# 1단계: 모델의 원본 출력을 먼저 확인
raw = (prompt | det_model).invoke(
    {"question": "봉준호 감독의 2006년 한국 영화 '괴물'의 제목, 개봉연도, 장르들, 대략적 평점을 알려줘."}
)
print(raw.content)

In [ ]:
# 2단계: JSON 문자열 → Pydantic 객체
# (0.5B 소형 모델이라 가끔 형식이 어긋날 수 있습니다. 실패하면 위 셀을 다시 실행해 보세요.)
result = parser.parse(raw.content)
print(result)
print(f"제목: {result.title}, 연도: {result.year}")

# Agent

지금까지의 체인(Chain)은 **우리가 정한 순서**대로 실행됐습니다.
Agent는 반대로, **LLM이 스스로 판단**해서 어떤 도구를 언제 쓸지 결정합니다.

```
질문 → [LLM: 생각] → 도구 선택·실행 → 결과 관찰 → [LLM: 생각] → ... → 최종 답변
```

Agent를 만들려면 두 가지가 필요합니다.
1. **Tool**: LLM이 쓸 수 있는 도구 (이름 + 설명 + 실행 함수)
2. **루프**: LLM 출력에서 도구 호출을 읽어 실행하고, 결과를 다시 LLM에게 돌려주는 반복 구조

## Tool 정의 — `@tool` 데코레이터

일반 파이썬 함수에 `@tool`을 붙이면 LangChain 도구가 됩니다.
**docstring이 곧 도구 설명서**가 되어 LLM에게 전달되므로, 언제·어떻게 쓰는 도구인지 명확히 적어야 합니다.

In [ ]:
import re
from langchain_core.tools import tool

In [ ]:
@tool("multiply")
def multiply_tool(expr: str) -> str:
    """
    두 수의 곱을 계산합니다.
    입력 형식 예: "12 7", "12,7", "12 x 7"
    """
    nums = re.findall(r"-?\d+(?:\.\d+)?", expr)
    if len(nums) < 2:
        return "오류: 두 개의 숫자가 필요합니다. 예: '12 7'"
    x, y = float(nums[0]), float(nums[1])
    return str(x * y)

In [ ]:
CITY_TO_COUNTRY = {
    "Seoul": "South Korea",
    "Tokyo": "Japan",
    "Paris": "France",
}

@tool("lookup_country")
def lookup_country_tool(city_text: str) -> str:
    """
    도시가 속한 국가를 알려줍니다. (모르는 도시는 'unknown')
    입력 예: "Paris", "Seoul"
    """
    m = re.search(r"[A-Za-z]+", city_text)
    city = m.group(0) if m else city_text.strip().split()[0]
    return CITY_TO_COUNTRY.get(city, "unknown")

In [ ]:
# 도구는 이름, 설명, 입력 스키마를 가진 객체가 됩니다.
print(f"name        = {multiply_tool.name}")
print(f"description = {multiply_tool.description.strip()}")
print(f"args        = {multiply_tool.args}")

# 도구도 Runnable이므로 invoke()로 직접 실행할 수 있습니다.
print(f"실행 결과   = {multiply_tool.invoke('12 x 7')}")

## 문서 검색 Tool — RAG를 도구로 만들기 (Chroma)

108 노트북에서 배운 RAG 파이프라인(분할 → 임베딩 → Chroma → retriever)을
그대로 **Agent의 도구 하나**로 포장합니다.
그러면 Agent가 "이 질문은 문서를 찾아봐야겠다"라고 판단할 때만 검색을 수행합니다.

실습용으로 가상의 보안 제품 **NeoGuard Pro**의 사내 문서 3건을 사용합니다.
(파일 업로드 없이 노트북 안에서 바로 만듭니다.)

In [ ]:
from langchain_core.documents import Document

raw_docs = [
    Document(
        page_content=(
            "NeoGuard Pro 제품 개요\n"
            "NeoGuard Pro는 중소기업용 엔드포인트 보안 솔루션입니다.\n"
            "최신 버전은 3.2이며, 2026년 5월에 출시되었습니다.\n"
            "주요 기능: 실시간 악성코드 탐지, 랜섬웨어 차단, 중앙 관리 콘솔.\n"
            "지원 운영체제: Windows 10 이상, Ubuntu 22.04 이상."
        ),
        metadata={"source": "product_overview"},
    ),
    Document(
        page_content=(
            "보안 권고문 NG-2026-01\n"
            "NeoGuard Pro 3.0 이하 버전에서 권한 상승 취약점이 발견되었습니다.\n"
            "CVSS 점수는 7.8(High)입니다.\n"
            "공격자는 이 취약점을 이용해 관리자 권한을 획득할 수 있습니다.\n"
            "대응 방안: 3.1 이상 버전으로 즉시 업데이트하십시오."
        ),
        metadata={"source": "advisory_NG-2026-01"},
    ),
    Document(
        page_content=(
            "기술 지원 정책\n"
            "NeoGuard Pro의 기술 지원 기간은 출시 후 3년입니다.\n"
            "일반 문의는 support@neoguard.example 로 접수합니다.\n"
            "긴급 보안 문의는 24시간 핫라인(1588-0000)을 이용하십시오."
        ),
        metadata={"source": "support_policy"},
    ),
]

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
docs = text_splitter.split_documents(raw_docs)

MODEL_EMBED = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embedding = HuggingFaceEmbeddings(model_name=MODEL_EMBED)

db = Chroma.from_documents(docs, embedding)
retriever = db.as_retriever(search_kwargs={"k": 2})

In [ ]:
@tool("search_docs")
def search_docs_tool(query: str) -> str:
    """
    사내 문서(NeoGuard 제품 소개, 보안 권고문, 지원 정책)에서 관련 내용을 검색합니다.
    NeoGuard 제품에 대한 질문에 사용하세요.
    """
    found = retriever.invoke(query)
    return "\n---\n".join(d.page_content for d in found)

In [ ]:
# 도구 단독 테스트
print(search_docs_tool.invoke("NeoGuard Pro 최신 버전"))

In [ ]:
# Agent에게 제공할 전체 도구 목록
TOOLS = [multiply_tool, lookup_country_tool, search_docs_tool]

## ReAct — 추론(Reason)과 행동(Act)을 번갈아 수행

ReAct는 가장 기본적인 Agent 패턴입니다. LLM이 정해진 형식으로 출력하면,
우리 코드가 그 형식을 읽어(파싱) 도구를 대신 실행해 줍니다.

```
Question:     답해야 할 질문
Thought:      무엇을 해야 할지 생각          ← LLM이 작성
Action:       사용할 도구 이름               ← LLM이 작성
Action Input: 도구에 전달할 입력             ← LLM이 작성
Observation:  도구 실행 결과                 ← 우리 코드가 채움!
...           (필요한 만큼 반복)
Final Answer: 최종 답변                      ← LLM이 작성
```

핵심 규칙 두 가지:
- `Observation:`은 반드시 **실제 도구 실행 결과**여야 합니다. 그래서 모델 생성 시 `stop=["Observation:"]`으로 잘라, 모델이 결과를 지어내지 못하게 합니다.
- 지금까지의 기록(scratchpad)을 매번 프롬프트 뒤에 붙여, 모델이 이전 단계를 기억하게 합니다.

In [ ]:
REACT_PROMPT = """당신은 도구를 사용할 수 있는 AI 에이전트입니다.

사용 가능한 도구:
{tools}

반드시 아래 형식으로만 답변하세요.

Question: 답해야 할 질문
Thought: 지금 무엇을 해야 하는지 생각합니다.
Action: 사용할 도구 이름. 반드시 [{tool_names}] 중 하나
Action Input: 도구에 전달할 입력
Observation: 도구 실행 결과
...(Thought/Action/Action Input/Observation은 여러 번 반복될 수 있습니다)
Thought: 이제 최종 답을 알겠습니다.
Final Answer: 질문에 대한 최종 답변 (한국어)

Question: {question}
{scratchpad}"""

In [ ]:
def run_react(question, tools, llm, max_steps=5, verbose=True):
    """ReAct 루프: 생성 → 파싱 → 도구 실행 → Observation 추가 → 반복."""
    tool_map = {t.name: t for t in tools}
    tools_desc = "\n".join(f"- {t.name}: {t.description.strip()}" for t in tools)
    tool_names = ", ".join(tool_map)
    scratchpad = ""

    for step in range(1, max_steps + 1):
        prompt_text = REACT_PROMPT.format(
            tools=tools_desc,
            tool_names=tool_names,
            question=question,
            scratchpad=scratchpad,
        )

        # stop=["Observation:"] : 모델이 도구 결과를 지어내지 못하게 차단
        text = llm.invoke(prompt_text, stop=["Observation:"]).content.strip()

        if verbose:
            print(f"\n===== [Step {step}] 모델 출력 =====")
            print(text)

        # 1) 최종 답변이 나왔는가?
        m_final = re.search(r"Final Answer:\s*(.*)", text, re.DOTALL)
        if m_final:
            return m_final.group(1).strip()

        # 2) 도구 호출 파싱
        m_act = re.search(r"Action:\s*(.+)", text)
        m_inp = re.search(r"Action Input:\s*(.+)", text)
        if not (m_act and m_inp):
            return text  # 형식을 지키지 못하면 출력 그대로 반환

        name = m_act.group(1).strip().strip("[]`\"'")
        arg = m_inp.group(1).strip().strip("`\"'")

        # 3) 도구 실행 → Observation
        if name in tool_map:
            obs = tool_map[name].invoke(arg)
        else:
            obs = f"오류: '{name}' 도구는 없습니다. [{tool_names}] 중에서 선택하세요."

        if verbose:
            print(f"----- [Step {step}] Observation -----")
            print(obs)

        # 4) 기록을 쌓고 다음 단계로
        scratchpad += f"{text}\nObservation: {obs}\nThought: "

    return "최대 단계 수를 초과했습니다."

In [ ]:
# 예제 1: 계산 도구 하나만 필요한 질문
answer = run_react("12와 7을 곱하면 얼마인가요?", TOOLS, det_model)
print(f"\n>>> 최종 답변: {answer}")

In [ ]:
# 예제 2: 문서 검색(RAG) 도구가 필요한 질문
answer = run_react("NeoGuard Pro의 최신 버전은 무엇인가요?", TOOLS, det_model)
print(f"\n>>> 최종 답변: {answer}")

In [ ]:
# 예제 3: 두 개의 도구를 차례로 사용해야 하는 질문
answer = run_react(
    "NeoGuard Pro 3.0 이하 버전 취약점의 CVSS 점수를 찾고, Seoul이 어느 나라인지도 알려주세요.",
    TOOLS, det_model,
)
print(f"\n>>> 최종 답변: {answer}")

> **소형 모델 참고**: 0.5B 모델은 가끔 형식을 벗어나거나 도구 선택을 틀립니다.
> 실패하면 셀을 다시 실행하거나 질문을 더 단순하게 바꿔 보세요.
> 실무에서는 tool calling을 정식 지원하는 더 큰 모델을 쓰지만, **동작 원리는 지금 구현한 루프와 동일**합니다.

## Conversational — 대화 기록으로 맥락 유지

Agent가 여러 턴의 대화를 이어가려면 이전 대화를 기억해야 합니다.
`RunnableWithMessageHistory`는 체인 앞뒤에 **대화 기록 저장/주입**을 자동으로 붙여 줍니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.output_parsers import StrOutputParser

In [ ]:
history_store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    """세션 ID별로 대화 기록을 따로 보관합니다."""
    if session_id not in history_store:
        history_store[session_id] = InMemoryChatMessageHistory()
    return history_store[session_id]

In [ ]:
conv_prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 유용하고 간결한 한국어 어시스턴트야."),
    MessagesPlaceholder("history"),
    ("human", "{input}"),
])

base_chain = conv_prompt | chat_model | StrOutputParser()

In [ ]:
conv_agent = RunnableWithMessageHistory(
    base_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

session_cfg = {"configurable": {"session_id": "demo-user-1"}}

In [ ]:
print("[+] 첫 번째 질문")
print(conv_agent.invoke({"input": "RAG가 무엇인지 두 문장으로 설명해 주세요."}, config=session_cfg))

print("\n[+] 두 번째 질문 (이전 대화를 기억해야 답할 수 있음)")
print(conv_agent.invoke({"input": "방금 설명한 내용을 한 문장으로 줄여 주세요."}, config=session_cfg))

In [ ]:
# 저장된 대화 기록 확인
for msg in history_store["demo-user-1"].messages:
    print(f"[{type(msg).__name__}] {msg.content[:60]}...")

## Planning — 계획을 먼저 세우고 실행 (Plan-and-Execute)

복잡한 목표는 한 번의 ReAct로 풀기 어렵습니다.
Plan-and-Execute는 역할을 둘로 나눕니다.

| 역할 | 하는 일 |
|---|---|
| **Planner** | 목표를 실행 가능한 단계 목록(JSON)으로 분해 |
| **Executor** | 각 단계를 ReAct 루프로 하나씩 해결 |

마지막에 단계별 결과를 모아 최종 답변을 정리합니다.
(예전에는 `langchain_experimental`의 `PlanAndExecute`를 썼지만 지금은 중단되었으므로, 직접 구현합니다. 구조를 이해하기에도 이 편이 좋습니다.)

In [ ]:
import json
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
PLANNER_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "너는 계획 수립 보조자야. 사용자의 목표를 달성하기 위한 "
     "간단하고 실행 가능한 단계만 JSON 배열로 출력해. "
     "설명이나 코드블록 없이 JSON만. 각 원소는 한국어 한 문장으로."),
    ("human",
     "목표: {goal}\n"
     '출력 형식 예시: ["12와 7의 곱을 계산한다", "Paris가 속한 국가를 조회한다"]'),
])

planner = PLANNER_PROMPT | det_model | StrOutputParser()

In [ ]:
def make_plan(goal: str) -> list:
    """Planner: 목표 → 단계 목록(JSON 배열)."""
    plan_text = planner.invoke({"goal": goal}).strip()
    plan_text = plan_text.strip("`").removeprefix("json").strip()  # 코드블록 제거
    try:
        steps = json.loads(plan_text)
        assert isinstance(steps, list) and all(isinstance(s, str) for s in steps)
    except Exception:
        steps = [goal]  # JSON 파싱 실패 시: 목표 전체를 단일 단계로
    return steps

In [ ]:
# Planner 단독 테스트
plan = make_plan("1) 15와 4를 곱하고 2) NeoGuard Pro의 최신 버전을 확인해서 정리")
for i, step in enumerate(plan, 1):
    print(f"{i}. {step}")

In [ ]:
SUMMARY_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "단계별 실행 결과를 바탕으로 목표에 대한 최종 답변을 한국어 한두 문장으로 작성해."),
    ("human", "목표: {goal}\n\n단계별 결과:\n{observations}"),
])

summarizer = SUMMARY_PROMPT | det_model | StrOutputParser()

In [ ]:
def plan_and_execute(goal: str, verbose=True):
    """Planner → (단계마다 ReAct Executor) → 최종 요약."""
    # (a) 계획 수립
    steps = make_plan(goal)
    if verbose:
        print("===== 계획 =====")
        for i, s in enumerate(steps, 1):
            print(f"{i}. {s}")

    # (b) 단계별 실행 (Executor = ReAct)
    observations = []
    for i, step in enumerate(steps, 1):
        if verbose:
            print(f"\n########## 단계 {i} 실행: {step} ##########")
        result = run_react(step, TOOLS, det_model, verbose=verbose)
        observations.append(f"[{i}] {step} -> {result}")

    # (c) 최종 요약
    final = summarizer.invoke({"goal": goal, "observations": "\n".join(observations)})
    return {"plan": steps, "observations": observations, "output": final}

In [ ]:
res = plan_and_execute("1) 15와 4를 곱하고 2) NeoGuard Pro의 최신 버전을 확인해서 한 문장으로 정리")

In [ ]:
print("===== [Plan-and-Execute 결과] =====")
print("계획:", res["plan"])
print("단계별 결과:", *res["observations"], sep="\n  ")
print("\n최종 답변:", res["output"])

## 정리

| 패턴 | 핵심 아이디어 | 이 노트북의 구현 |
|---|---|---|
| **ReAct** | Thought → Action → Observation 반복, LLM이 도구를 스스로 선택 | `run_react()` 루프 |
| **Conversational** | 대화 기록을 프롬프트에 주입해 맥락 유지 | `RunnableWithMessageHistory` |
| **Planning** | 계획 수립(Planner)과 실행(Executor)의 역할 분리 | `make_plan()` + `plan_and_execute()` |

**핵심 개념 세 가지**
1. Agent = LLM(판단) + Tool(실행) + 루프(반복 구조). 마법이 아니라 파싱과 반복문입니다.
2. 도구의 **docstring(설명)** 이 좋을수록 LLM이 도구를 올바르게 선택합니다.
3. RAG도 Agent 입장에서는 `search_docs`라는 **도구 하나**일 뿐입니다. 필요할 때만 검색합니다.

작은 0.5B 모델로도 Agent의 동작 원리를 그대로 재현할 수 있다는 것,
그리고 상용 프레임워크(LangGraph, tool calling)도 내부적으로는 이 루프를 정교하게 만든 것뿐이라는 점이 이번 실습의 핵심입니다.